# LIME explanations for AgroScan

LIME perturbs superpixels of one leaf image and fits a simple local model to show **which regions pushed the prediction toward a class**. It uses the same preprocessing as the API (via `models/model_meta.json`), so the explanation is about the model that is actually served.

**Run top to bottom** (`Kernel > Restart & Run All`) or headless: `jupyter nbconvert --to notebook --execute notebooks/lime_example.ipynb`.

Read explanations as *evidence about the model*, not proof of agronomic reasoning: a highlighted background or soil patch is a red flag for shortcut learning.

In [ ]:
# --- Parameters (edit these, nothing else needs changing) ---
IMAGE_DIR = "data/plantdoc_test"   # any folder of leaf images; read-only. Run `dvc pull` first if empty.
N_IMAGES = 4
SEED = 0                            # same seed -> same images -> repeatable
OUT_DIR = "reports/explainability"

In [ ]:
import os
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "models").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "notebooks"))
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from explain_utils import load_assets, display_image, make_predict_fn, pick_images  # noqa: E402
np.random.seed(SEED)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
model, meta, class_names = load_assets()
predict = make_predict_fn(model, meta)
paths = pick_images(IMAGE_DIR, N_IMAGES, SEED)
assert paths, f"No images in {IMAGE_DIR}: run `dvc pull` or change IMAGE_DIR"
print(f"model_version={meta.get('version')}  img_size={meta['img_size']}  images={len(paths)}")

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

explainer = lime_image.LimeImageExplainer(random_state=SEED)
fig, axes = plt.subplots(len(paths), 2, figsize=(7, 3.4 * len(paths)))
axes = np.atleast_2d(axes)
for row, p in zip(axes, paths):
    img = display_image(p, meta)
    probs = predict(img[None])[0]
    top = int(np.argmax(probs))
    exp = explainer.explain_instance(img, predict, top_labels=3, hide_color=0,
                                     num_samples=1000, random_seed=SEED, batch_size=50)
    lime_img, mask = exp.get_image_and_mask(top, positive_only=True, num_features=5, hide_rest=False)
    row[0].imshow(img)
    row[0].set_title(p.parent.name + "/" + p.name, fontsize=8)
    row[0].axis("off")
    row[1].imshow(mark_boundaries(lime_img / 255.0, mask))
    row[1].set_title(f"LIME: {class_names[top]} ({probs[top]:.2f})", fontsize=8)
    row[1].axis("off")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/lime_examples.png", dpi=150)
plt.show()

## What to check
- Does the highlighted region cover the **lesion / leaf tissue**, not the background?
- Are wrong predictions explained by a visually similar disease (e.g. early vs late blight)?
- Record 2-3 observations in `MODEL_CARD.md` (limits section).